In [2]:
!pip install -U -q "mineru[core]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.6/786.6 kB 16.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 90.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.3/98.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 112.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
from pathlib import Path

INPUT_DIR = Path("/kaggle/input/datasets/octave2402/filepdf")

OUTPUT_DIR = Path("/kaggle/working/md_files")
TMP_DIR = Path("/kaggle/temp/pdf_tmp")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(INPUT_DIR.rglob("*.pdf")) if INPUT_DIR.exists() else []
print(len(pdfs))


28


In [ ]:
import os
import subprocess
import shutil
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

log_file = OUTPUT_DIR / "run.log"

print("🚀 Inizio conversione PARALLELA (2 GPU) con MinerU...\n")
print("=" * 60)

def elabora_singolo_pdf(pdf, gpu_id):
    # 1. Salta i file di servizio
    if pdf.name.startswith("0_"):
        return f"⏩ SKIP {pdf.name} (file 0_ escluso)"

    # 2. Ricrea la struttura logica
    relative_pdf = pdf.relative_to(INPUT_DIR)
    target_dir = OUTPUT_DIR / relative_pdf.parent
    target_dir.mkdir(parents=True, exist_ok=True)

    name = pdf.stem
    
    # 3. Percorso di output di MinerU
    out_md = target_dir / name / "hybrid_auto" / f"{name}.md"

    if out_md.exists():
        return f"⏩ SKIP {relative_pdf} (già convertito)"

    # Avviso di partenza (indica su quale GPU sta girando)
    print(f"▶️ INIZIO: {pdf.name} (Assegnato a GPU {gpu_id})")

    # 4. Copia temporanea
    src = TMP_DIR / relative_pdf
    src.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pdf, src)

    t0 = time.time()

    # 5. MAGIA MULTI-GPU: Creiamo un ambiente isolato per il processo
    # dicendogli di "vedere" solo la GPU che gli abbiamo assegnato (0 o 1)
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

    # Esecuzione (l'output viene catturato in background per non sporcare lo schermo)
    r = subprocess.run(
        [
            "mineru", 
            "-p", str(src),
            "-o", str(target_dir),
            "-m", "auto"
        ],
        capture_output=True, text=True, env=env
    )

    elapsed = time.time() - t0
    ok = (r.returncode == 0 and out_md.exists())

    # 6. Pulizia
    src.unlink(missing_ok=True)

    # 7. Prepara il resoconto finale per questo PDF
    if ok:
        msg = f"✅ {relative_pdf} COMPLETATO su GPU {gpu_id} in {elapsed:.0f}s"
    else:
        msg = f"❌ {relative_pdf} FALLITO su GPU {gpu_id} (Codice: {r.returncode})\nErrore log: {r.stderr[-500:]}"
    
    # Scrive nel file di log condiviso
    with open(log_file, "a") as f:
        f.write(msg + "\n")
        
    return msg

# ==========================================
# GESTORE MULTIPROCESSO
# ==========================================
# max_workers=2 garantisce che partano esattamente 2 PDF alla volta
with ThreadPoolExecutor(max_workers=2) as executor:
    
    # Invia tutti i PDF all'esecutore, alternando GPU 0 e GPU 1
    # i % 2 genera la sequenza: 0, 1, 0, 1, 0, 1...
    future_to_pdf = {executor.submit(elabora_singolo_pdf, pdf, i % 2): pdf for i, pdf in enumerate(pdfs)}
    
    # Man mano che i PDF finiscono la conversione, stampa il risultato a schermo
    for future in as_completed(future_to_pdf):
        risultato = future.result()
        print(risultato)
        print("-" * 60)

print("\n🎉 ELABORAZIONE PARALLELA COMPLETATA! Markdown e Immagini sono in:", OUTPUT_DIR)

🚀 Inizio conversione PARALLELA (2 GPU) con MinerU...

⏩ SKIP Probability,_Random_Variables_and_Stochastic_Processes/10_Random_Walks_and_Other_Applications.pdf (già convertito)
------------------------------------------------------------
⏩ SKIP 0_indice_analitico.pdf (file 0_ escluso)
------------------------------------------------------------
⏩ SKIP 0_indice_generale.pdf (file 0_ escluso)
------------------------------------------------------------
⏩ SKIP Probability,_Random_Variables_and_Stochastic_Processes/11_Spectral_Representation.pdf (già convertito)
------------------------------------------------------------
⏩ SKIP Probability,_Random_Variables_and_Stochastic_Processes/13_Mean_Square_Estimation.pdf (già convertito)
------------------------------------------------------------
⏩ SKIP Probability,_Random_Variables_and_Stochastic_Processes/14_Entropy.pdf (già convertito)
------------------------------------------------------------
⏩ SKIP Probability,_Random_Variables_and_Stochasti

In [ ]:
import shutil

print("Sto comprimendo l'intero pacchetto (Markdown + Immagini ritagliate)...")
shutil.make_archive("/kaggle/working/risultati_mineru", 'zip', "/kaggle/working/md_files")
print("✓ Fatto! Vai nel pannello di destra 'Output' per scaricare: risultati_mineru.zip")

In [ ]:
import os
from IPython.display import FileLink

# Assicuriamoci di essere nella cartella corretta
os.chdir('/kaggle/working')

# Genera un link cliccabile per il download
display(FileLink('risultati_mineru.zip'))